In [1]:
# Cell 1: install dependencies
import transformers, peft, accelerate
print(transformers.__version__)

5.0.0


In [2]:
# Cell 2: imports and config
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

warnings.filterwarnings("ignore")

SCORE_COLS  = ("cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions")
SCORE_RANGE = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
BASE_MODEL  = "/kaggle/input/models/limyi9/qwen2-5-1-5b-instruct/transformers/default/1"
LORA_PATH   = "/kaggle/input/models/limyi9/scorelora/transformers/default/1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cuda


In [3]:
# Cell 3: load model
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    local_files_only=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
print("model loaded.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded.


In [4]:
# Cell 4: load test data
test_df = pd.read_csv("/kaggle/input/competitions/feedback-prize-english-language-learning/test.csv")
print(f"test samples: {len(test_df)}")
print(test_df.head())

test samples: 3
        text_id                                          full_text
0  0000C359D63E  when a person has no experience on a job their...
1  000BAD50D026  Do you think students would benefit from being...
2  00367BB2546B  Thomas Jefferson once states that "it is wonde...


In [5]:
# Cell 5: inference and submission
def get_essay_prompt(essay):
    essay = essay.strip()[:2000]
    prompt_text = (
        "Rate this English essay on 6 dimensions (1.0-5.0 in 0.5 steps).\n"
        'Output ONLY JSON: {"cohesion":x,"syntax":x,"vocabulary":x,'
        '"phraseology":x,"grammar":x,"conventions":x}\n'
        f"Essay:\n{essay}"
    )
    return [{"role": "user", "content": prompt_text}]

def clamp_score(v):
    return min(SCORE_RANGE, key=lambda x: abs(x - v))

def parse_json(text):
    try:
        match = re.search(r"\{[^{}]+\}", text, re.DOTALL)
        if match:
            data = json.loads(match.group())
            return {c: clamp_score(float(data.get(c, 3.0))) for c in SCORE_COLS}
    except Exception:
        pass
    return {c: 3.0 for c in SCORE_COLS}

@torch.no_grad()
def generate_score(essay):
    messages = get_essay_prompt(essay)
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80,
        do_sample=False,
    )
    gen_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return parse_json(gen_text)

print("running inference on test set...")
test_results = []
for i, row in enumerate(test_df.itertuples(), 1):
    res = generate_score(row.full_text)
    test_results.append({"text_id": row.text_id, **res})
    if i % 10 == 0:
        print(f"  {i}/{len(test_df)}")

sub_df = pd.DataFrame(test_results)
sub_df.to_csv("/kaggle/working/submission.csv", index=False)
print("done. submission.csv saved.")
print(sub_df.head())

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


running inference on test set...
done. submission.csv saved.
        text_id  cohesion  syntax  vocabulary  phraseology  grammar  \
0  0000C359D63E       3.0     2.5         3.0          2.5      2.5   
1  000BAD50D026       2.5     3.0         3.0          2.5      2.5   
2  00367BB2546B       3.5     4.0         3.5          3.5      3.5   

   conventions  
0          2.5  
1          2.5  
2          3.5  
